# 02 · Conditional edges and loops

After this notebook you can route with `add_conditional_edges`, build a retry loop that always terminates, stop runaway graphs with `recursion_limit`, route from inside a node with `Command`, and fan work out in parallel with `Send`.

**Time:** ~30 min · **Runs:** offline on CPU in < 30 s, no LLM needed · **Needs:** [01 · State, nodes and edges](01_state_nodes_and_edges.ipynb)

## Why this matters in interviews

- Branches and loops are where agents get both their power and their bills. "How do you stop an agent looping forever?" expects **layered guards** and a **give-up path**, not "set max iterations to 50".
- Self-correction ("reflection") is a standard design question. The strong answer is *loop against a verifier* (a parser, a test suite), with a capped number of retries whose value you have measured.
- Map-reduce shows up in RAG and research agents ("search five sources in parallel"). You should know how the branches are created, how results are merged, and what happens when one branch fails.

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `ag16` | How do you stop an agent looping forever? |
| `ag10` | What is reflection, and what does it cost? |
| `ag31` | What is a router agent, and why is it the highest-leverage pattern? |
| `ag32` | How do you parallelise agent work safely? |
| `ag34` | How do you decide the step budget for an agent? |
| `ag44` | How do you decide when a task is "done"? |

## 1. Setup

Everything here is plain Python, so every run gives the same output. Where a real system would call an LLM, a scripted function stands in and the text says so.

In [ ]:
import json
import operator
import time
from typing import Annotated, Literal, TypedDict

import matplotlib.pyplot as plt
import numpy as np
from langgraph.errors import GraphRecursionError
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command, RetryPolicy, Send

print("imports ok")

The graph-drawing helper from notebook 01 (solid = fixed edge, dashed = conditional, arcs below = loops back).

In [ ]:
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

def draw_graph(app, title=None, xray=False):
    """Draw a compiled LangGraph left to right: solid = fixed edge, dashed = conditional edge,
    arcs above = skip ahead, arcs below = loop back. Red text = the path-map label."""
    g = app.get_graph(xray=xray)
    edges = [(e.source, e.target, e.conditional, e.data) for e in g.edges]
    col, frontier = {"__start__": 0}, ["__start__"]
    while frontier:                                     # column = hops from START (BFS)
        nxt = []
        for n in frontier:
            for s, t, _, _ in edges:
                if s == n and t not in col and t != "__end__":
                    col[t] = col[n] + 1; nxt.append(t)
        frontier = nxt
    for n in g.nodes:
        col.setdefault(n, max(col.values()) + 1)
    col["__end__"] = max(c for n, c in col.items() if n != "__end__") + 1
    rows = {}
    for n in g.nodes:
        rows.setdefault(col[n], []).append(n)
    pos = {n: (c * 2.4, (len(ns) - 1) / 2 - i) for c, ns in rows.items() for i, n in enumerate(ns)}
    nrow = max(len(v) for v in rows.values())
    fig, ax = plt.subplots(figsize=(min(11, 2.0 * len(rows) + 0.6), min(4.5, 0.9 * nrow + 1.4)))
    boxes = {}
    for n, (x, y) in pos.items():
        parts = [{"__start__": "START", "__end__": "END"}.get(p, p) for p in n.split(":")]
        fc = {"START": "#d3f9d8", "END": "#ffe3e3"}.get(parts[-1], "#e7f0ff")
        boxes[n] = FancyBboxPatch((x - 0.8, y - 0.28), 1.6, 0.56, boxstyle="round,pad=0.05", fc=fc, ec="#3b5bdb")
        ax.add_patch(boxes[n])
        ax.text(x, y, "\n".join(parts), ha="center", va="center", fontsize=8.5 if len(parts) == 1 else 7.5)
    pairs = {(s, t) for s, t, _, _ in edges}
    for s, t, cond, data in edges:
        (x1, y1), (x2, y2) = pos[s], pos[t]
        span = col[t] - col[s]
        if (t, s) in pairs and abs(span) == 1:
            rad = 0.2                                   # two-way pair: split the arrows apart
        elif span == 1 or (span == 0 and s != t):
            rad = 0.0
        else:
            rad = -0.3 if span > 1 else -0.45           # skip ahead above, loop back below
        ax.add_patch(FancyArrowPatch((x1, y1), (x2, y2), patchA=boxes[s], patchB=boxes[t], arrowstyle="-|>",
                                     mutation_scale=12, lw=1.3, color="#495057", linestyle="--" if cond else "-",
                                     connectionstyle=f"arc3,rad={rad}"))
        if data is not None and str(data) != t:
            mx, my = (x1 + x2) / 2 + 0.5 * rad * (y2 - y1), (y1 + y2) / 2 - 0.5 * rad * (x2 - x1)
            ax.text(mx, my + 0.1, str(data), ha="center", fontsize=7.5, color="#c92a2a")
    xs = [p[0] for p in pos.values()]; ys = [p[1] for p in pos.values()]
    ax.set_xlim(min(xs) - 1.0, max(xs) + 1.0); ax.set_ylim(min(ys) - 0.75, max(ys) + 0.85)
    ax.axis("off"); ax.set_title(title or "graph", fontsize=10)
    plt.tight_layout(); plt.show()

## 2. Branching: `add_conditional_edges`

**In plain English:** after a node finishes, a small **routing function** looks at the state and returns a label. A **path map** translates each label to the node that should run next. This is the router pattern: the cheapest, highest-leverage piece of most agent systems, because it sends easy requests down cheap paths.

```python
builder.add_conditional_edges(source_node, routing_function, {label: next_node, ...})
```

In production the routing function often reads a label produced by an LLM classifier; here a keyword rule plays that part.

In [ ]:
class Ticket(TypedDict):
    text: str
    category: str
    handled_by: str

def classify(state: Ticket) -> dict:
    t = state["text"].lower()
    if any(w in t for w in ("refund", "invoice", "charged")):
        return {"category": "billing"}
    if any(w in t for w in ("error", "crash", "bug")):
        return {"category": "technical"}
    return {"category": "general"}

def route_ticket(state: Ticket) -> Literal["billing", "technical", "general"]:
    return state["category"]                       # a label, not a node name

rb = StateGraph(Ticket)
rb.add_node("classify", classify)
rb.add_node("billing_team", lambda s: {"handled_by": "billing team (human)"})
rb.add_node("tech_team", lambda s: {"handled_by": "on-call engineer"})
rb.add_node("faq_bot", lambda s: {"handled_by": "FAQ bot (cheap model)"})
rb.add_edge(START, "classify")
rb.add_conditional_edges("classify", route_ticket,
                         {"billing": "billing_team", "technical": "tech_team", "general": "faq_bot"})
for n in ("billing_team", "tech_team", "faq_bot"):
    rb.add_edge(n, END)
router = rb.compile()

for text in ["I was charged twice", "The app crashes on start", "Do you ship to Canada?"]:
    out = router.invoke({"text": text})
    print(f"{text:28s} -> {out['category']:9s} -> {out['handled_by']}")
assert router.invoke({"text": "refund please"})["handled_by"].startswith("billing")

In [ ]:
draw_graph(router, "Router: one conditional edge, three destinations (red = path-map labels)")

Two details worth knowing:

- **Without a path map** the routing function must return node names directly, and you should pass the possible destinations (`add_conditional_edges("classify", fn, ["billing_team", "tech_team", "faq_bot"])`) or annotate the return type with `Literal[...]`, otherwise the graph cannot know the edges in advance for drawing and validation.
- The routing function can return **a list of node names** to run several branches in parallel. Section 6 takes that further with `Send`.

## 3. Loops: generate → check → retry until it passes (or give up)

**In plain English:** the model writes something, a checker verifies it, and if it fails we send the error back and try again. The two things that make this safe are a **verifier that is not the model** (here a JSON parser plus a Pydantic schema) and a **maximum number of tries** followed by an explicit **give-up path**.

The generator below is a scripted stand-in for an LLM that fixes its output when shown the error, reproducing three failures real models produce: JSON wrapped in a Markdown fence, a missing field, then a correct answer.

In [ ]:
from pydantic import BaseModel, ValidationError

class Contact(BaseModel):
    name: str
    email: str

SCRIPTED_ATTEMPTS = [
    '```json\n{"name": "Mamitha", "email": "mamitha@example.com"}\n```',   # fenced: not valid JSON
    '{"name": "Mamitha"}',                                                # valid JSON, missing a field
    '{"name": "Mamitha", "email": "mamitha@example.com"}',                  # correct
]

class GenState(TypedDict):
    task: str
    draft: str
    error: str
    tries: int
    max_tries: int
    status: str
    feedback: Annotated[list, operator.add]     # what we would send back to the model

def generate(state: GenState) -> dict:
    # A real node: llm.invoke(task + "\nYour last attempt failed: " + state["error"])
    attempt = SCRIPTED_ATTEMPTS[min(state["tries"], len(SCRIPTED_ATTEMPTS) - 1)]
    return {"draft": attempt, "tries": state["tries"] + 1}

def check(state: GenState) -> dict:
    try:
        Contact.model_validate_json(state["draft"])
        return {"error": ""}
    except ValidationError as e:
        err = e.errors()[0]
        msg = f"{err['type']} at {'.'.join(map(str, err['loc'])) or 'root'}: {err['msg']}"
        return {"error": msg, "feedback": [msg]}

The routing function is where termination lives. Note the order: success first, then the budget, then retry.

In [ ]:
def after_check(state: GenState) -> Literal["done", "give_up", "retry"]:
    if not state["error"]:
        return "done"
    if state["tries"] >= state["max_tries"]:
        return "give_up"
    return "retry"

def finish(state: GenState) -> dict:
    return {"status": f"ok after {state['tries']} tries"}

def give_up(state: GenState) -> dict:           # a named outcome, not an exception
    return {"status": f"gave up after {state['tries']} tries; last error: {state['error']}"}

lb = StateGraph(GenState)
lb.add_node("generate", generate)
lb.add_node("check", check)
lb.add_node("finish", finish)
lb.add_node("give_up", give_up)
lb.add_edge(START, "generate")
lb.add_edge("generate", "check")
lb.add_conditional_edges("check", after_check, {"done": "finish", "give_up": "give_up", "retry": "generate"})
lb.add_edge("finish", END)
lb.add_edge("give_up", END)
fixer = lb.compile()

def run_fixer(max_tries):
    return fixer.invoke({"task": "Extract the contact as JSON", "tries": 0, "max_tries": max_tries,
                         "error": "", "draft": "", "status": "", "feedback": []})

In [ ]:
for update in fixer.stream({"task": "Extract the contact as JSON", "tries": 0, "max_tries": 5, "error": "",
                            "draft": "", "status": "", "feedback": []}, stream_mode="updates"):
    for node, delta in update.items():
        shown = {k: v for k, v in delta.items() if k != "feedback"}
        print(f"{node:9s} {shown}")

ok, capped = run_fixer(max_tries=5), run_fixer(max_tries=2)
print("\nmax_tries=5 ->", ok["status"])
print("max_tries=2 ->", capped["status"])
assert ok["status"] == "ok after 3 tries" and capped["status"].startswith("gave up after 2")
assert json.loads(ok["draft"]) == {"name": "Mamitha", "email": "mamitha@example.com"}

The `feedback` list is exactly what a real node would append to the prompt on the next attempt: a precise, machine-produced error ("Field required at email"), which is far more useful to a model than "that was wrong, try again".

Now look at the drawing. `draw_ascii()` **does not show the loop-back edge** (a limitation of its layout), even though the edge exists:

In [ ]:
print(fixer.get_graph().draw_ascii())
loop_edges = [(e.source, e.target, e.data) for e in fixer.get_graph().edges if e.source == "check"]
print("edges leaving 'check':", loop_edges)
assert ("check", "generate", "retry") in loop_edges

In [ ]:
draw_graph(fixer, "Self-correcting loop: check -> generate is the retry edge")

## 4. The safety net: `recursion_limit` and `GraphRecursionError`

**In plain English:** a loop with a bug (a checker that can never pass, a router that never says "done") would run forever and spend money forever. LangGraph counts **super-steps** and aborts with `GraphRecursionError` when a run reaches `recursion_limit`. Treat it like a circuit breaker: it should never trip in normal operation, and your graph should end itself through a give-up path long before it.

**Set it explicitly on every call.** Older LangGraph releases defaulted to 25; the version installed here (1.2) defaults to 10,007 (overridable with the `LANGGRAPH_DEFAULT_RECURSION_LIMIT` environment variable), which is effectively "no limit" for a loop that calls a paid model on every step. Tutorials that say "the default is 25" are out of date.

Here is a loop with no budget in the router and a generator that is stuck on the same broken output:

In [ ]:
stuck_calls = {"generate": 0}

def stuck_generate(state: GenState) -> dict:
    stuck_calls["generate"] += 1
    return {"draft": SCRIPTED_ATTEMPTS[0], "tries": state["tries"] + 1}   # always the fenced JSON

def no_budget_router(state: GenState) -> Literal["done", "retry"]:
    return "done" if not state["error"] else "retry"            # BUG: no max_tries check

sb = StateGraph(GenState)
sb.add_node("generate", stuck_generate)
sb.add_node("check", check)
sb.add_edge(START, "generate"); sb.add_edge("generate", "check")
sb.add_conditional_edges("check", no_budget_router, {"done": END, "retry": "generate"})
runaway = sb.compile()

try:
    runaway.invoke({"task": "x", "tries": 0, "max_tries": 0, "error": "", "draft": "", "status": "",
                    "feedback": []}, {"recursion_limit": 10})
except GraphRecursionError as e:
    print("GraphRecursionError:", str(e).splitlines()[0])
print("generate ran", stuck_calls["generate"], "times before the breaker tripped")
assert stuck_calls["generate"] == 5

How the count works: every node execution in sequence is one super-step, and the graph's internal `__start__` step counts too, so a run that executes N nodes one after another needs `recursion_limit >= N + 1`. Above, a limit of 10 left room for nine node steps: generate and check four times over, then a fifth generate. The rule, checked on a plain 3-node chain:

In [ ]:
chain = StateGraph(GenState)
chain.add_sequence([("a", lambda s: {}), ("b", lambda s: {}), ("c", lambda s: {})])
chain.add_edge(START, "a")
chain = chain.compile()

def runs_with(limit):
    try:
        chain.invoke({"tries": 0}, {"recursion_limit": limit}); return True
    except GraphRecursionError:
        return False

print({limit: runs_with(limit) for limit in (2, 3, 4, 5)})
assert not runs_with(3) and runs_with(4)          # 3 nodes need a limit of 4

Better than crashing at the limit: let the graph **see its remaining budget** and wrap up. Declare a `RemainingSteps` key in the state; LangGraph fills it in (it is a *managed value*, you never write it). This is how the prebuilt ReAct agent returns "Sorry, need more steps" instead of raising.

In [ ]:
from langgraph.managed import RemainingSteps

class BudgetState(GenState):
    remaining_steps: RemainingSteps

def budget_router(state: BudgetState) -> Literal["done", "wrap_up", "retry"]:
    if not state["error"]:
        return "done"
    return "wrap_up" if state["remaining_steps"] <= 2 else "retry"

bb = StateGraph(BudgetState)
bb.add_node("generate", stuck_generate); bb.add_node("check", check); bb.add_node("wrap_up", give_up)
bb.add_edge(START, "generate"); bb.add_edge("generate", "check")
bb.add_conditional_edges("check", budget_router, {"done": END, "wrap_up": "wrap_up", "retry": "generate"})
bb.add_edge("wrap_up", END)
out = bb.compile().invoke({"task": "x", "tries": 0, "max_tries": 0, "error": "", "draft": "", "status": "",
                           "feedback": []}, {"recursion_limit": 10})
print("status:", out["status"])
assert out["status"].startswith("gave up")

| Guard | Where it lives | What the user sees | Use it as |
|---|---|---|---|
| `tries >= max_tries` in the router | your state + routing function | a graceful, named outcome | **the primary control** |
| `RemainingSteps` managed value | state key, filled by LangGraph | a graceful wrap-up near the limit | a generic budget for any loop |
| `recursion_limit` (set it: the 1.2 default is 10,007) | the `config` of each call | an exception (`GraphRecursionError`) | **the backstop** that should never trip |
| wall-clock timeout / cost ceiling | your code around the call, or node `timeout=` | a degraded answer | protection against slow or expensive steps |
| repeat / no-progress detection | the router compares the new error with the last one | early give-up | stops a loop that is not learning (exercise 2) |

## 5. `Command`: update the state *and* choose the next node in one return

**In plain English:** with conditional edges, one function does the work and a separate function decides where to go. With `Command`, the node returns both at once: `Command(update={...}, goto="next_node")`. It is the natural fit when the decision comes out of the same step as the work (an LLM that classifies *and* extracts), and it is how agents hand off to each other (notebook 06) and how interrupted runs are resumed (notebook 04).

In [ ]:
def triage(state: Ticket) -> Command[Literal["billing_team", "tech_team", "faq_bot"]]:
    category = classify(state)["category"]
    destination = {"billing": "billing_team", "technical": "tech_team", "general": "faq_bot"}[category]
    return Command(update={"category": category}, goto=destination)

cb = StateGraph(Ticket)
cb.add_node("triage", triage)                   # no add_conditional_edges needed
cb.add_node("billing_team", lambda s: {"handled_by": "billing team (human)"})
cb.add_node("tech_team", lambda s: {"handled_by": "on-call engineer"})
cb.add_node("faq_bot", lambda s: {"handled_by": "FAQ bot (cheap model)"})
cb.add_edge(START, "triage")
for n in ("billing_team", "tech_team", "faq_bot"):
    cb.add_edge(n, END)
command_router = cb.compile()

for text in ["I was charged twice", "Found a bug in export"]:
    out = command_router.invoke({"text": text})
    print(f"{text:24s} -> {out['category']:9s} -> {out['handled_by']}")
assert command_router.invoke({"text": "bug"})["handled_by"] == router.invoke({"text": "bug"})["handled_by"]

The return annotation `Command[Literal[...]]` is not decoration: it is how LangGraph learns the possible destinations for drawing and validation. Leave it out and the run still works, but the graph structure is wrong:

In [ ]:
def triage_no_hint(state: Ticket):
    return triage(state)                       # same behaviour, no annotation

nb = StateGraph(Ticket)
nb.add_node("triage", triage_no_hint)
for n in ("billing_team", "tech_team", "faq_bot"):
    nb.add_node(n, lambda s: {"handled_by": "someone"}); nb.add_edge(n, END)
nb.add_edge(START, "triage")
print("edges without the hint:", sorted((e.source, e.target) for e in nb.compile().get_graph().edges
                                         if e.source == "triage"))
print("edges with the hint   :", sorted((e.source, e.target) for e in command_router.get_graph().edges
                                         if e.source == "triage"))
draw_graph(command_router, "Command router: the destinations come from the Command[Literal[...]] hint")

| | `add_conditional_edges` | `Command` returned by a node |
|---|---|---|
| Where the decision lives | a separate routing function | inside the node that did the work |
| State update + routing | two functions | one return value |
| How the graph knows the edges | the path map (or a list of destinations) | the `Command[Literal[...]]` annotation (or `destinations=`) |
| Typical use | routers, retry loops, `tools_condition` | agent handoffs, resuming after `interrupt()`, jumping out of a subgraph (`graph=Command.PARENT`) |

## 6. Parallel fan-out and fan-in with `Send` (map-reduce)

**In plain English:** sometimes you only know at run time how many parallel jobs there are: one per document, per sub-question, per supplier. A routing function returns a list of `Send("node", payload)`; each `Send` starts its own copy of the node with its own private input. All copies run in the same super-step, a reducer on the output key collects their results, and the next node (the fan-in) runs once, after all of them have finished.

```
                       ┌─► score_review({"review": r1}) ─┐
 START ─► fan_out ─────┼─► score_review({"review": r2}) ─┼─► aggregate ─► END
   (returns [Send, ...]) └─► score_review({"review": r3}) ─┘   (runs once; results merged by operator.add)
```

Each worker sleeps 0.2 s to stand in for an LLM call, so we can see whether the branches really overlap.

In [ ]:
REVIEWS = ["Great product, fast delivery", "Terrible support, still waiting",
           "Good value but slow shipping", "Excellent quality, would buy again", "Broken on arrival, awful"]
POS, NEG = {"great", "fast", "good", "excellent", "value"}, {"terrible", "slow", "broken", "awful", "waiting"}

class ReviewJob(TypedDict):                     # the private input of ONE branch
    review: str

class Batch(TypedDict):
    reviews: list
    scores: Annotated[list, operator.add]       # every branch appends one result
    summary: str

def fan_out(state: Batch) -> list[Send]:
    return [Send("score_review", {"review": r}) for r in state["reviews"]]

def score_review(job: ReviewJob) -> dict:
    time.sleep(0.2)                             # pretend this is an LLM call
    words = set(job["review"].lower().replace(",", "").split())
    return {"scores": [{"review": job["review"], "score": len(words & POS) - len(words & NEG)}]}

def aggregate(state: Batch) -> dict:
    mean = np.mean([s["score"] for s in state["scores"]])
    return {"summary": f"{len(state['scores'])} reviews, mean score {mean:+.2f}"}

mb = StateGraph(Batch)
mb.add_node("score_review", score_review)
mb.add_node("aggregate", aggregate)
mb.add_conditional_edges(START, fan_out, ["score_review"])
mb.add_edge("score_review", "aggregate")
mb.add_edge("aggregate", END)
mapreduce = mb.compile()

out = mapreduce.invoke({"reviews": REVIEWS})
for s in out["scores"]:
    print(f"{s['score']:+d}  {s['review']}")
print(out["summary"])
assert len(out["scores"]) == len(REVIEWS)

In [ ]:
draw_graph(mapreduce, "Map-reduce: one Send per review (dashed), one aggregate at the end")

How much does the fan-out buy? Compare the default (all branches at once) with a concurrency cap, set per call with `config={"max_concurrency": k}`. A cap is how you avoid hammering a rate-limited API.

In [ ]:
mapreduce.invoke({"reviews": REVIEWS[:1]})                     # warm-up (thread pool start-up)
timings = {}
for label, cfg in [("all at once", {}), ("max_concurrency=2", {"max_concurrency": 2}),
                   ("max_concurrency=1", {"max_concurrency": 1})]:
    t0 = time.perf_counter()
    mapreduce.invoke({"reviews": REVIEWS}, cfg)
    timings[label] = time.perf_counter() - t0
    print(f"{label:18s} {timings[label]:.2f} s")
assert timings["all at once"] < 0.6 * timings["max_concurrency=1"]

fig, ax = plt.subplots(figsize=(7, 2.8))
ax.barh(list(timings), list(timings.values()), color=["#2f9e44", "#f59f00", "#c92a2a"])
ax.axvline(0.2 * len(REVIEWS), ls="--", color="gray", label="5 x 0.2 s done one by one")
ax.set_xlabel("wall-clock seconds for 5 reviews"); ax.set_title("Send branches run concurrently (one super-step)")
ax.legend(loc="lower right"); plt.tight_layout(); plt.show()

**What if one branch fails?** By default an exception in any branch fails the whole super-step, and so the whole run. Two tools:

1. `retry_policy=RetryPolicy(...)` on the node retries **transient** errors with exponential backoff. Check what it retries by default: `ConnectionError` and HTTP 5xx, but *not* `ValueError`, `OSError` or `TimeoutError` (a subclass of `OSError`). Pass `retry_on=` to change that.
2. For errors that retrying won't fix, **catch inside the worker and return an error record**, so the fan-in can report "3 of 4 done, 1 failed" instead of failing everything or, worse, silently averaging fewer items. Decide up front whether partial success is acceptable.

In [ ]:
from langgraph.types import default_retry_on
print("retried by default? ConnectionError:", default_retry_on(ConnectionError()),
      "| TimeoutError:", default_retry_on(TimeoutError()), "| ValueError:", default_retry_on(ValueError()))
assert default_retry_on(ConnectionError()) and not default_retry_on(TimeoutError())

flaky_calls = {"n": 0}

def score_review_safely(job: ReviewJob) -> dict:
    if job["review"] == "":                        # a corrupt input: retrying won't fix it
        return {"scores": [{"review": "", "error": "empty review"}]}
    flaky_calls["n"] += 1
    if flaky_calls["n"] == 2:                      # one transient failure
        raise TimeoutError("model timed out")
    return score_review(job)

def aggregate_partial(state: Batch) -> dict:
    good = [s for s in state["scores"] if "error" not in s]
    failed = [s["error"] for s in state["scores"] if "error" in s]
    return {"summary": f"{len(good)} of {len(state['scores'])} scored, {len(failed)} failed: {failed}"}

pb = StateGraph(Batch)
pb.add_node("score_review", score_review_safely,
            retry_policy=RetryPolicy(max_attempts=3, initial_interval=0.01, retry_on=TimeoutError))
pb.add_node("aggregate", aggregate_partial)
pb.add_conditional_edges(START, fan_out, ["score_review"])
pb.add_edge("score_review", "aggregate"); pb.add_edge("aggregate", END)
partial = pb.compile().invoke({"reviews": REVIEWS[:3] + [""]}, {"max_concurrency": 1})
print(partial["summary"], "| worker calls incl. the retried one:", flaky_calls["n"])
assert partial["summary"].startswith("3 of 4 scored, 1 failed") and flaky_calls["n"] == 4

## 7. How many retries are worth paying for?

Each retry is another model call. Real traffic mixes tasks that a retry can fix with tasks that can **never** pass (the information is missing, a tool is broken). A quick simulation with the loop graph from section 3: 80% of tasks succeed with probability 0.6 per attempt, 20% can never pass. The same 200 tasks (with the same pre-drawn attempt outcomes) are run under each `max_tries` setting, and we record the success rate and the average number of generator calls (your cost).

In [ ]:
class SimState(TypedDict):
    task_id: int
    tries: int
    max_tries: int
    ok: bool

P_SUCCESS, HOPELESS, N_TASKS = 0.6, 0.2, 200
draw = np.random.default_rng(0)
hopeless = draw.random(N_TASKS) < HOPELESS                  # these tasks can never pass
attempt_ok = draw.random((N_TASKS, 6)) < P_SUCCESS          # would attempt j of task i pass?

def sim_generate(s: SimState) -> dict:
    i, j = s["task_id"], s["tries"]
    return {"tries": j + 1, "ok": bool(attempt_ok[i, j] and not hopeless[i])}

sim = StateGraph(SimState)
sim.add_node("generate", sim_generate)
sim.add_edge(START, "generate")
sim.add_conditional_edges("generate", lambda s: "stop" if s["ok"] or s["tries"] >= s["max_tries"] else "retry",
                          {"stop": END, "retry": "generate"})
sim = sim.compile()

settings, success, calls = list(range(1, 7)), [], []
for k in settings:
    runs = [sim.invoke({"task_id": i, "tries": 0, "max_tries": k, "ok": False}) for i in range(N_TASKS)]
    success.append(np.mean([r["ok"] for r in runs]))
    calls.append(np.mean([r["tries"] for r in runs]))
    print(f"max_tries={k}: success {success[-1]:.2f}, avg calls per task {calls[-1]:.2f}")

theory = [(1 - HOPELESS) * (1 - (1 - P_SUCCESS) ** k) for k in settings]
assert max(abs(s - t) for s, t in zip(success, theory)) < 0.1
assert success[-1] < 0.9 and calls[-1] > calls[2] + 0.3      # plateau in success, cost still rising

In [ ]:
fig, ax1 = plt.subplots(figsize=(8.5, 3.6))
ax1.plot(settings, success, "o-", color="#2f9e44", label="success rate (simulated)")
ax1.plot(settings, theory, ":", color="#2f9e44", label="success rate (theory)")
ceiling = 1 - hopeless.mean()
ax1.axhline(ceiling, color="gray", lw=0.8, ls="--")
ax1.text(1.05, ceiling + 0.01, f"ceiling: {hopeless.mean():.0%} of these tasks can never pass", fontsize=8, color="gray")
ax1.set_xlabel("max_tries"); ax1.set_ylabel("success rate"); ax1.set_ylim(0.4, 0.9)
ax2 = ax1.twinx()
ax2.plot(settings, calls, "s--", color="#c92a2a", label="avg generator calls (cost)")
ax2.set_ylabel("average calls per task")
ax1.set_title("More retries: success plateaus, cost keeps rising (hopeless tasks burn every try)")
fig.legend(loc="lower right", bbox_to_anchor=(0.87, 0.2), fontsize=8)
plt.tight_layout(); plt.show()

Going from 1 to 3 or 4 tries buys almost all of the available success; after that the green curve sits on its ceiling while the red cost line keeps climbing, because every hopeless task burns the full budget. That is why the step budget is set from the distribution of **successful** runs (around their p95), why a no-progress check (exercise 2) pays for itself, and why you alarm on cost per request rather than on total spend.

## Try it yourself

**1.** Add an `"escalate"` branch to the router in section 2: tickets containing "lawyer" or "legal" go to a `legal_team` node, whatever else they mention. Make the check happen *before* the billing rule.

In [ ]:
# Solution — try it yourself first, then run this
def classify_with_legal(state: Ticket) -> dict:
    if any(w in state["text"].lower() for w in ("lawyer", "legal")):
        return {"category": "legal"}
    return classify(state)

eb = StateGraph(Ticket)
eb.add_node("classify", classify_with_legal)
for n, who in [("billing_team", "billing"), ("tech_team", "engineer"), ("faq_bot", "bot"), ("legal_team", "legal")]:
    eb.add_node(n, lambda s, who=who: {"handled_by": who}); eb.add_edge(n, END)
eb.add_edge(START, "classify")
eb.add_conditional_edges("classify", lambda s: s["category"], {"billing": "billing_team", "technical": "tech_team",
                                                               "general": "faq_bot", "legal": "legal_team"})
esc = eb.compile()
print(esc.invoke({"text": "Refund me or I call my lawyer"})["handled_by"])
assert esc.invoke({"text": "Refund me or I call my lawyer"})["handled_by"] == "legal"
assert esc.invoke({"text": "Refund me"})["handled_by"] == "billing"

**2.** No-progress detection: change the router of the self-correcting loop so it gives up early when the **same error appears twice in a row**, even if tries remain. Test it with the stuck generator from section 4 and `max_tries=10`: it should stop after 2 tries.

In [ ]:
# Solution — try it yourself first, then run this
def router_no_progress(state: GenState) -> Literal["done", "give_up", "retry"]:
    if not state["error"]:
        return "done"
    repeated = len(state["feedback"]) >= 2 and state["feedback"][-1] == state["feedback"][-2]
    if repeated or state["tries"] >= state["max_tries"]:
        return "give_up"
    return "retry"

npb = StateGraph(GenState)
npb.add_node("generate", stuck_generate); npb.add_node("check", check); npb.add_node("give_up", give_up)
npb.add_edge(START, "generate"); npb.add_edge("generate", "check")
npb.add_conditional_edges("check", router_no_progress, {"done": END, "give_up": "give_up", "retry": "generate"})
npb.add_edge("give_up", END)
res = npb.compile().invoke({"task": "x", "tries": 0, "max_tries": 10, "error": "", "draft": "", "status": "",
                            "feedback": []})
print(res["status"])
assert res["tries"] == 2

**3.** A self-correcting loop against a **unit test** instead of a parser. The scripted "model" proposes three versions of `is_palindrome`; the checker runs the tests and feeds back the first failure. Build the loop and print which attempt passes.

In [ ]:
# Solution — try it yourself first, then run this
CODE_ATTEMPTS = [
    "def is_palindrome(s):\n    return s == s[::-1]",
    "def is_palindrome(s):\n    s = s.lower()\n    return s == s[::-1]",
    "def is_palindrome(s):\n    t = [c for c in s.lower() if c.isalnum()]\n    return t == t[::-1]",
]
TESTS = [("racecar", True), ("Racecar", True), ("Was it a car or a cat I saw?", True), ("hello", False)]

def gen_code(state: GenState) -> dict:
    return {"draft": CODE_ATTEMPTS[min(state["tries"], 2)], "tries": state["tries"] + 1}

def run_tests(state: GenState) -> dict:
    scope = {}
    exec(state["draft"], scope)          # fine for our own scripted strings; sandbox real model output!
    for arg, want in TESTS:
        if scope["is_palindrome"](arg) != want:
            msg = f"is_palindrome({arg!r}) should be {want}"
            return {"error": msg, "feedback": [msg]}
    return {"error": ""}

tb = StateGraph(GenState)
tb.add_node("generate", gen_code); tb.add_node("check", run_tests)
tb.add_node("finish", finish); tb.add_node("give_up", give_up)
tb.add_edge(START, "generate"); tb.add_edge("generate", "check")
tb.add_conditional_edges("check", after_check, {"done": "finish", "give_up": "give_up", "retry": "generate"})
tb.add_edge("finish", END); tb.add_edge("give_up", END)
res = tb.compile().invoke({"task": "write is_palindrome", "tries": 0, "max_tries": 4, "error": "",
                           "draft": "", "status": "", "feedback": []})
print(res["status"]); print("feedback sent back:", res["feedback"])
assert res["status"] == "ok after 3 tries" and len(res["feedback"]) == 2

## Say it in an interview

**30-second answer (stopping loops):** "Layered guards. The router owns termination: success first, then a step budget from state, then retry; every exit is a named outcome, including 'gave up, here is the last error'. A no-progress check stops a loop that repeats the same failure. `recursion_limit` is only the backstop that turns a bug into an exception instead of an invoice, and I'd alarm if it ever trips. Around the whole run I keep a wall-clock and a cost ceiling."

**30-second answer (reflection):** "Loop against a verifier, not against the model's own opinion: a JSON parser, a schema, a test suite, a citation check. Feed the concrete error back, cap it at two or three retries, and measure whether retry two actually moves the metric. Each retry is another full model call."

**Facts to have ready:**
- `add_conditional_edges(source, fn, path_map)`: `fn` returns a label, the map turns it into a node. Annotate or list destinations so the graph can be drawn and validated.
- `recursion_limit` counts super-steps and raises `GraphRecursionError`. Older releases defaulted to 25, LangGraph 1.2 to 10,007, so pass it explicitly in `config`. A `RemainingSteps` state key lets a node wrap up before the limit.
- `Command(update=..., goto=...)` routes from inside a node; annotate `-> Command[Literal[...]]`.
- `Send("node", payload)` creates one branch per item with its own input; a reducer (`operator.add`) merges the results; the fan-in node runs once after all branches. `max_concurrency` caps parallelism, `RetryPolicy` retries transient failures.

**Follow-ups they ask:** *"What happens if one branch of a map-reduce fails?"* The super-step fails unless you catch it; decide whether partial results are acceptable and tell the next step which branches are missing. *"How do you pick the step budget?"* From the distribution of successful runs (around p95), then watch how many runs end at the cap: if many do, a tool or task is broken, and raising the cap only makes failures more expensive.

**Traps:** putting the budget check after the retry branch (it never fires), relying on `recursion_limit` as the only guard, retrying non-retryable errors (a 422 will fail the same way every time), and letting parallel branches write the same key without a reducer.

## Next

- [03 · Checkpointing, memory and time travel](03_checkpointing_memory_and_time_travel.ipynb): persist the state of these loops and rewind them.
- [05 · ReAct tool-calling agent](05_react_tool_calling_agent.ipynb): the most famous loop of all, with a model choosing the edge.
- [ReAct and plan-and-execute](../15_agentic_ai/03_react_and_plan_and_execute.ipynb) · [retries, backoff and circuit breakers](../16_production/02_retries_backoff_rate_limits_circuit_breakers.ipynb) · [async concurrency for LLM calls](../16_production/01_async_concurrency_for_llm_calls.ipynb)
- Theory: [LangGraph course](../../langgraph/index.html) (chapters 4–5), [agentic AI flow](../../agentic_ai_flow/index.html), [interview bank](../../ai_interview_prep/index.html).